# LAMU 2026 + Pytania z Ksiezyca - wspolna mapa pytan

### Czym jest ten notatnik?

Ten notatnik laczy dwa zrodla pytan:

1. **LAMU 2026** - pytania zadane przez dzieci w ramach Letniej Akademii Mlodych Umyslow (projekt Radia Naukowego), pogrupowane w kategorie tematyczne.
2. **Pytania z Ksiezyca** - pytania zadane przez sluchaczy audycji w Radiu 357.

Oba zbiory zostaja **wspolnie zakodowane przez ten sam model jezykowy** i zrzutowane na **wspolna mape UMAP**, dzieki czemu mozna porownac obie kolekcje pytan w jednej przestrzeni semantycznej. Punkty sa pokolorowane wedlug zrodla (LAMU vs Pytania z Ksiezyca), zeby zobaczyc, w ktorych obszarach tematycznych oba zbiory sie pokrywaja, a gdzie sie roznia.

### Jak uruchomic ten notatnik?

1. Otworz strone [Google Colab](https://colab.research.google.com/)
2. Wybierz **Plik > Otworz notatnik > Przeslij** i wgraj ten plik (.ipynb)
3. W menu u gory wybierz **Srodowisko uruchomieniowe > Zmien typ srodowiska** i ustaw **GPU** (typ T4) - dzieki temu obliczenia beda szybsze
4. Kliknij **Srodowisko uruchomieniowe > Uruchom wszystko** (lub Ctrl+F9)
5. Poczekaj kilka minut - notatnik sam pobierze oba zrodla, zainstaluje biblioteki, zakoduje pytania i wygeneruje wykresy
6. Na koncu automatycznie pobierze sie plik HTML z interaktywnym wykresem

### Co robi kazdy krok?

| Krok | Co sie dzieje | Ile trwa |
|------|--------------|----------|
| 1. Instalacja bibliotek | Pobiera narzedzia potrzebne do analizy | ~1 min |
| 2. Pobranie zrodel | Sciaga PDF LAMU i plik tekstowy Pytan z Ksiezyca | kilka sekund |
| 3. Ekstrakcja pytan | Parsuje PDF LAMU i wczytuje pytania z Ksiezyca | natychmiastowe |
| 4. Embeddingi | Model jezykowy zamienia kazde pytanie w wektor 768 liczb | ~3-4 min (GPU) dla ~1500 pytan |
| 5. Wspolna redukcja UMAP | Sciaga 768 wymiarow do 2 dla wszystkich pytan jednoczesnie | ~30 sek |
| 6. Wykresy | Rysuje statyczny i interaktywny wykres z dwoma kolorami | natychmiastowe |
| 7. Eksport HTML | Zapisuje wynik do pobrania | natychmiastowe |

In [ ]:
!pip install -q sentence-transformers umap-learn plotly pymupdf requests

In [ ]:
import fitz  # pymupdf
import re
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

## 1. Pobranie obu zrodel danych

In [ ]:
PDF_URL = "https://radionaukowe.pl/wp-content/uploads/2026/03/LAMU-zadane-pytania-2026.pdf"
PDF_PATH = "LAMU-zadane-pytania-2026.pdf"

KSIEZYC_URL = "https://gist.githubusercontent.com/BERENZ/a2f58c81f5983e8f6b9e5191be4bad8a/raw/0fee1aec9d5ec6769ebe6e20bdeb0600ee0be692/pytania-z-ksiezyca.txt"
KSIEZYC_PATH = "pytania-z-ksiezyca.txt"

for url, path in [(PDF_URL, PDF_PATH), (KSIEZYC_URL, KSIEZYC_PATH)]:
    r = requests.get(url)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    print(f"Pobrano: {path} ({len(r.content) / 1024:.1f} KB)")

## 2. Ekstrakcja pytan z obu zrodel

In [ ]:
def extract_lamu_questions(pdf_path):
    """Extract questions grouped by topic from the LAMU PDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()

    topic_pattern = r'(FIZYKA|BIOLOGIA|WSZECH[\u015aS]WIAT|CZ[\u0141L]OWIEK|TECHNOLOGIE|ZIEMIA|MATEMATYKA|HISTORIA|CHEMIA)'
    parts = re.split(topic_pattern, full_text)

    questions = []
    current_topic = None
    topic_display = {
        'FIZYKA': 'Fizyka',
        'BIOLOGIA': 'Biologia',
        'WSZECHSWIAT': 'Wszech\u015bwiat',
        'CZLOWIEK': 'Cz\u0142owiek',
        'TECHNOLOGIE': 'Technologie',
        'ZIEMIA': 'Ziemia',
        'MATEMATYKA': 'Matematyka',
        'HISTORIA': 'Historia',
        'CHEMIA': 'Chemia'
    }

    for part in parts:
        part_stripped = part.strip()
        if re.match(topic_pattern, part_stripped):
            normalized = part_stripped.replace('\u015a', 'S').replace('\u0141', 'L')
            current_topic = topic_display.get(normalized, normalized)
            continue

        if current_topic is None:
            continue

        lines = part.split('\n')
        buffer = ""
        for line in lines:
            line = line.strip()
            if not line or line.startswith('LAMU') or line.startswith('Radio') or 'RADIO' in line.upper() or 'NAUKOWE' in line.upper() or 'LETNIA' in line.upper() or 'AKADEMIA' in line.upper():
                continue
            if re.match(r'^[\u2013\-]\s', line):
                if buffer:
                    questions.append((current_topic, buffer.strip()))
                buffer = re.sub(r'^[\u2013\-]\s*', '', line)
            else:
                if buffer:
                    buffer += ' ' + line

        if buffer:
            questions.append((current_topic, buffer.strip()))

    return questions

lamu_questions = extract_lamu_questions(PDF_PATH)
df_lamu = pd.DataFrame(lamu_questions, columns=['topic', 'question'])
df_lamu['source'] = 'LAMU 2026'
print(f"LAMU 2026 - pytan: {len(df_lamu)}")
print(df_lamu['topic'].value_counts())

In [ ]:
# Pytania z Ksiezyca - kazda niepusta linia to jedno pytanie
with open(KSIEZYC_PATH, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

ksiezyc_questions = [line.strip() for line in raw_lines if line.strip()]

df_ksiezyc = pd.DataFrame({'question': ksiezyc_questions})
df_ksiezyc['topic'] = '(brak kategorii)'
df_ksiezyc['source'] = 'Pytania z Ksi\u0119\u017cyca'
print(f"Pytania z Ksiezyca - pytan: {len(df_ksiezyc)}")

In [ ]:
# Polaczenie obu zbiorow
df = pd.concat([df_lamu, df_ksiezyc], ignore_index=True)
print(f"Wszystkich pytan: {len(df)}")
print(df['source'].value_counts())

## 3. Kodowanie pytan za pomoca polskich embeddingów

In [ ]:
MODEL_NAME = "sdadas/st-polish-paraphrase-from-distilroberta"

model = SentenceTransformer(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Wymiar embeddingów: {model.get_sentence_embedding_dimension()}")

In [ ]:
embeddings = model.encode(df['question'].tolist(), show_progress_bar=True, batch_size=32)
print(f"Ksztalt macierzy embeddingów: {embeddings.shape}")

## 4. Wspolna redukcja wymiarowosci (UMAP)

Oba zbiory pytan sa **razem** podawane do UMAP - dzieki temu wszystkie pytania lezą w tej samej przestrzeni 2D i mozna je bezposrednio porownac.

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

coords_2d = reducer.fit_transform(embeddings)
df['x'] = coords_2d[:, 0]
df['y'] = coords_2d[:, 1]
print(f"Redukcja wymiarów: {embeddings.shape[1]} -> 2 (na lacznie {len(df)} pytaniach)")

## 5. Wizualizacja

### 5a. Wykres statyczny (matplotlib)

In [ ]:
source_colors = {
    'LAMU 2026': '#d62728',                # czerwony
    'Pytania z Ksi\u0119\u017cyca': '#1f4e79'  # ciemny granat
}

fig, ax = plt.subplots(figsize=(14, 10))

# Rysujemy Ksiezyc najpierw (wieksza grupa) zeby LAMU bylo na wierzchu
for source in ['Pytania z Ksi\u0119\u017cyca', 'LAMU 2026']:
    mask = df['source'] == source
    ax.scatter(
        df.loc[mask, 'x'], df.loc[mask, 'y'],
        label=f'{source} (N={mask.sum()})',
        color=source_colors[source],
        s=25, alpha=0.55, edgecolors='white', linewidth=0.3
    )

ax.legend(title='Zr\u00f3d\u0142o', fontsize=11, title_fontsize=12,
          loc='best', framealpha=0.9)
ax.set_title('LAMU 2026 + Pytania z Ksiezyca - wspolna mapa embeddingów\n'
             f'(model: {MODEL_NAME}, redukcja: UMAP, N = {len(df)})', fontsize=14)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('combined_questions_static.png', dpi=150, bbox_inches='tight')
plt.show()
print("Zapisano: combined_questions_static.png")

### 5b. Wykres interaktywny (Plotly)

In [ ]:
df['question_short'] = df['question'].apply(
    lambda q: q[:80] + '...' if len(q) > 80 else q
)

fig = px.scatter(
    df, x='x', y='y',
    color='source',
    hover_data={'question': True, 'source': True, 'topic': True,
                'x': False, 'y': False, 'question_short': False},
    color_discrete_map=source_colors,
    title=f'LAMU 2026 + Pytania z Ksiezyca - wspolna mapa pytan (N = {len(df)})',
    labels={'source': 'Zr\u00f3d\u0142o', 'x': 'UMAP 1', 'y': 'UMAP 2'},
    width=950, height=720,
    category_orders={'source': ['Pytania z Ksi\u0119\u017cyca', 'LAMU 2026']}
)

fig.update_traces(
    marker=dict(size=7, opacity=0.6, line=dict(width=0.4, color='white')),
    hovertemplate='<b>%{customdata[1]}</b><br>kat.: %{customdata[2]}<br>%{customdata[0]}<extra></extra>'
)

fig.update_layout(
    legend_title_text='Zr\u00f3d\u0142o',
    font=dict(size=12),
    hoverlabel=dict(font_size=11),
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig.show()

### 5c. Statystyki

In [ ]:
print("=" * 50)
print("LAMU + Pytania z Ksiezyca - Podsumowanie")
print("=" * 50)
print(f"Calkowita liczba pytan: {len(df)}")
print(f"\nPodzial wedlug zrodla:")
for source, count in df['source'].value_counts().items():
    print(f"  {source}: {count}")
print(f"\nLAMU - rozklad kategorii:")
for topic, count in df_lamu['topic'].value_counts().items():
    print(f"  {topic}: {count}")
print(f"\nModel embeddingów: {MODEL_NAME}")
print(f"Wymiar embeddingów: {embeddings.shape[1]}")
print(f"Metoda redukcji: UMAP (n_neighbors=15, min_dist=0.1, metric=cosine)")

### 5d. Jak czytac ten wykres

---

Kazdy punkt to **jedno pytanie**. Kolor oznacza zrodlo:
- **Czerwony** - pytania dzieci z LAMU 2026
- **Granatowy** - pytania sluchaczy z audycji "Pytania z Ksiezyca" w Radiu 357

Obie kolekcje sa **wspolnie zrzutowane** na te sama mape UMAP - wiec odleglosci miedzy punktami sa porownywalne miedzy zbiorami. Im blizej siebie sa dwa punkty, tym bardziej semantycznie podobne sa pytania, niezaleznie od zrodla.

Interakcje z wykresem:
- **Najechanie myszka** na punkt - pokazuje zrodlo, kategorie (jesli z LAMU) i pelna tresc pytania.
- **Klikniecie zrodla w legendzie** - ukrywa/pokazuje dane zrodlo. Mozna np. zostawic tylko LAMU, zeby zobaczyc gdzie pytania dzieci sie skupiaja.
- **Zaznaczenie obszaru** (klik + przeciagniecie) - powiekszenie.

Co warto zauwazyc:
- **Obszary z mieszanymi kolorami** - tematy wspolne dla dzieci i sluchaczy radia (np. kosmos, zwierzeta, cialo czlowieka).
- **Obszary jednolicie granatowe** - tematy charakterystyczne dla sluchaczy doroslych (np. bardziej specjalistyczne pytania techniczne lub naukowe).
- **Obszary jednolicie czerwone** - tematy typowe dla dzieci (czesto pytania konkretne, codzienne, oparte na zmyslach lub bezposrednim doswiadczeniu).

UWAGA: w obu zbiorach uzyto **tego samego modelu embeddingów** i wspolnego UMAP. Bez tego porownanie nie byloby uczciwe - kazdy model i kazdy fit UMAP daje wlasna przestrzen.

## 6. Eksport do HTML (offline)

In [ ]:
HTML_PATH = "lamu_plus_ksiezyc.html"
PNG_PATH = "combined_questions_static.png"

fig.write_html(HTML_PATH, include_plotlyjs=True, full_html=True)
print(f"Zapisano interaktywny wykres: {HTML_PATH}")
print(f"Zapisano statyczny PNG:      {PNG_PATH}")

try:
    from google.colab import files
    files.download(HTML_PATH)
    files.download(PNG_PATH)
    print("Pobieranie plikow...")
except ImportError:
    print(f"Otworz plik {HTML_PATH} w przegladarce, aby zobaczyc wykres offline.")